# 📘 DATABASE MANAGEMENT AND DESIGN
### *A Practical Learning Series*

# Notebook 2.2 — Database Design
### From Schema Creation to Data Management

**By Amin Amirkhalili** — Business & Data Analyst

*Based on Chapter 2 (Section 2-4), Chapter 5 (Section 5-4), and Chapter 6 (Section 6-7) of:* **Database Management and Design** — Gove Allen, PhD · Gary Hansen, PhD · Robert Jackson, PhD


## 🗺️ Series Roadmap

This notebook is part of a practical learning series on database management and design. The series follows a logical progression — from understanding business needs, through conceptual modelling, all the way to writing SQL code on real databases.

| | |
|---|---|
| **1 — Fundamentals** | |
| Notebook 1 | Introduction: The Database Is the Heart of Information Systems |
| **2 — Database Management** | |
| Notebook 2.1 | Conceptual Data Model: From Conceptual Design to a Relational Model |
| **Notebook 2.2** | Database Design: From Schema Creation to Data Management **◀ You are here** |
| **3 — SQL & Data Analysis** | |
| Notebook 3.1 | Beginner Level: SELECT, FROM, WHERE, ORDER BY |
| Notebook 3.2 | Intermediate Level: Functions, GROUP BY, HAVING |
| Notebook 3.3 | Advanced Level I: Joining Tables — Inner & Outer Joins |
| Notebook 3.4 | Advanced Level II: Subqueries, Views, Temp Tables, CTEs |
| Notebook 3.5 | Data Cleaning with SQL *(coming soon)* |
| **4 — Data Mining** | |
| Notebook 4.1 | SQL in Data Mining *(coming soon)* |

## 📌 What This Notebook Covers

How to create database schemas using SQL, manage table structures, and manipulate data — hands-on. You will build the tables yourself, cell by cell.

## 💻 How to Use This Notebook

This is the **live practice version** — every query below runs for real.

1. Run the setup cell below **first** (▶ button on the left). It builds a small sample database with the same tables used in the series.
2. Run each query cell as you read. Change the queries — experiment!
3. To start fresh, just run the setup cell again.

> **Note on SQL dialects:** the reading copy of this notebook uses **SQL Server (SSMS)**. Here we run **SQLite** (built into Colab, no installation needed). The two are almost identical; wherever the syntax differs (e.g. `TOP` vs `LIMIT`, `+` vs `||` for text), a note shows both versions.

In [ ]:
#@title ▶️ Run this cell first — it prepares an empty practice database { display-mode: "form" }
# Notebook 2.2 is about CREATING tables from scratch — so we start with an
# empty database. q("...") runs SQL; SELECT results appear as a table.

import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run SQL. SELECT queries return a table; other commands report OK."""
    s = sql.strip().rstrip(";").strip()
    first = s.split()[0].upper() if s else ""
    if first in ("SELECT", "WITH", "PRAGMA"):
        return pd.read_sql_query(sql, conn)
    conn.executescript(sql)
    print("OK —", first, "executed.")

def tables():
    """List the tables that currently exist."""
    return pd.read_sql_query(
        "SELECT name AS TableName FROM sqlite_master WHERE type='table' ORDER BY name", conn)

print("✅ Empty practice database ready. Use q(\"...\") to run SQL and tables() to list tables.")


## Part 1 — Creating a Database Schema

At this stage, your conceptual model has already been designed and you create a relational database design based on that (see Notebook 2.1). Now it is time to turn that design into a real, working database. This is where theory becomes action.

### 1.1 The Database Hierarchy

- 🖥️ **Server** — the physical or virtual computer running the database engine (e.g., SQL Server).
- 🗄️ **Database** — a named container that holds all related tables and objects.
- 📋 **Tables** — structured sets of rows and columns, each representing one entity.

### 1.2 Creating the Database

In SQL Server (SSMS): `CREATE DATABASE SalesDB;`

*(In SQLite, a database is simply a file — Colab already created one for us in the setup cell, so we can go straight to creating tables.)*

SQL provides two main categories of commands:

| Category | Meaning |
|---|---|
| **DDL** | Data Definition Language — defines and changes the **structure** of tables (CREATE, ALTER, DROP). |
| **DML** | Data Manipulation Language — works with the **data** inside tables (INSERT, UPDATE, DELETE, SELECT). |

> ⚠️ **Important — there is no Undo in SQL.** When you drop a table or delete rows, they are gone — unless you have a backup or a transaction. Always double-check your WHERE clause before running DELETE or UPDATE.

## Part 2 — Managing Table Structures

### Creating a table from scratch

**Example 1 — a simple table:**

In [ ]:
q("""
CREATE TABLE Employee (
    EmployeeID INT NOT NULL,
    FirstName VARCHAR(20),
    LastName VARCHAR(50),
    Age INT,
    PRIMARY KEY (EmployeeID)
)
""")

In [ ]:
tables()

**Example 2 — a table with a foreign key** (we create the parent table Customer first):

In [ ]:
q("""
CREATE TABLE Customer (
    CustomerID INT NOT NULL,
    FirstName VARCHAR(20),
    LastName VARCHAR(50),
    PRIMARY KEY (CustomerID)
)
""")

In [ ]:
q("""
CREATE TABLE Sale (
    SaleID INT NOT NULL,
    SaleDate DATE NOT NULL,
    Tax NUMERIC(8,2) NOT NULL,
    Shipping NUMERIC(8,2) NOT NULL,
    CustomerID INT,
    PRIMARY KEY (SaleID),
    FOREIGN KEY (CustomerID)
        REFERENCES Customer(CustomerID) ON DELETE SET NULL
)
""")

**Example 3 — a composite primary key.** Three columns together uniquely identify each row:

In [ ]:
q("""
CREATE TABLE SaleItem (
    SaleID INT NOT NULL,
    ProductID INT NOT NULL,
    ItemSize NUMERIC(3,1) NOT NULL,
    Quantity INT NOT NULL,
    SalePrice NUMERIC(8,2) NOT NULL,
    PRIMARY KEY (SaleID, ProductID, ItemSize),
    FOREIGN KEY (SaleID)
        REFERENCES Sale(SaleID) ON DELETE CASCADE
)
""")

### Constraints: NOT NULL

The NOT NULL constraint means a column must always contain a value. Use it for any column that is essential to the row's identity. Try to break it — this insert should **fail** because EmployeeID is NOT NULL:

In [ ]:
try:
    q("INSERT INTO Employee (EmployeeID, FirstName) VALUES (NULL, 'Ghost')")
except Exception as e:
    print("❌ Rejected, as expected:", e)

### Common data types

| Data Type | Description |
|---|---|
| INT | Whole numbers (IDs, quantities, age). |
| VARCHAR(n) | Variable-length text up to n characters. |
| CHAR(n) | Fixed-length text. |
| NUMERIC(p, s) | Exact decimal with p digits, s decimals. Good for money. |
| DECIMAL(p, s) | Identical to NUMERIC. |
| DATE | A calendar date, no time component. |

### ON DELETE options for foreign keys

| Option | Behaviour |
|---|---|
| RESTRICT / NO ACTION | Blocks deleting a parent that still has children (default). |
| SET NULL | Parent deleted; child FK becomes NULL (FK must be nullable). |
| CASCADE | Deleting the parent deletes all its children. |
| SET DEFAULT | Child FK gets its default value. Rarely used. |

> 💡 **Key rule:** with ON DELETE SET NULL, the foreign key column must be nullable.

### Deleting a table

In [ ]:
q("""
DROP TABLE IF EXISTS EmployeeCopy
""")

> ⚠️ **Parent–child order matters.** If tables are linked by foreign keys, drop the child before the parent: SaleItem first, then Sale, then Customer.

### Copying a table

*(SQL Server: `SELECT * INTO EmployeeCopy FROM Employee`. SQLite writes it as `CREATE TABLE ... AS SELECT` — same result:)*

In [ ]:
q("""
INSERT INTO Employee VALUES
 (1, 'Amin', 'Amirkhalili', 40),
 (2, 'Sara', 'Lopez', 27),
 (3, 'Kim', 'Nguyen', 31),
 (4, 'John', 'Smith', 52)
""")

In [ ]:
q("""
DROP TABLE IF EXISTS EmployeeCopy;
CREATE TABLE EmployeeCopy AS
SELECT *
FROM Employee
WHERE Age > 30
""")

In [ ]:
q("""
SELECT * FROM EmployeeCopy
""")

> ⚠️ **Check before you copy.** The copy fails if a table with the same name already exists — always DROP TABLE IF EXISTS first.

### Modifying an existing table — ALTER TABLE

**Add a column:**

In [ ]:
q("""
ALTER TABLE Employee
ADD Education VARCHAR(20)
""")

In [ ]:
q("""
SELECT * FROM Employee
""")

**Remove a column:**

In [ ]:
q("""
ALTER TABLE Employee
DROP COLUMN Education
""")

**Change a column's data type** — SQL Server only (SQLite does not support ALTER COLUMN):

```sql
ALTER TABLE Customer
ALTER COLUMN LastName VARCHAR(100);
```

**Rename a column** (standard SQL, works here):

In [ ]:
q("""
ALTER TABLE EmployeeCopy
RENAME COLUMN Age TO EmployeeAge
""")

In [ ]:
q("""
SELECT * FROM EmployeeCopy
""")

**Rename a table** — standard SQL (works here):

In [ ]:
q("""
ALTER TABLE EmployeeCopy
RENAME TO Staff
""")

In [ ]:
tables()

SQL Server uses a stored procedure instead:

```sql
EXEC sp_rename 'EmployeeCopy', 'Staff';
```

> 💡 In SQL Server, primary/foreign keys added via ALTER TABLE are automatically NOT NULL. If you need a nullable foreign key, define it in CREATE TABLE.

## Part 3 — Managing Data Inside Tables

The three essential DML commands:

| Command | Purpose |
|---|---|
| INSERT INTO | Add new rows to a table. |
| UPDATE | Modify existing rows. |
| DELETE | Remove rows from a table. |

### 3.1 INSERT INTO — adding new rows

In [ ]:
q("""
INSERT INTO Employee (EmployeeID, FirstName, LastName, Age)
VALUES (5, 'Elli', 'Rahimi', 35)
""")

If you are inserting values for every column in order, you can omit the column list:

In [ ]:
q("""
INSERT INTO Employee
VALUES (6, 'Luke', 'Perry', 29)
""")

In [ ]:
q("""
SELECT * FROM Employee
""")

> ⚠️ **Column count must match.** The number of values must exactly match the number of columns. If a value is not available, use NULL (only valid for nullable columns).

### INSERT INTO ... SELECT — copying data from another table

Both tables must already exist before running this command:

In [ ]:
q("""
DROP TABLE IF EXISTS EmployeeNames;
CREATE TABLE EmployeeNames (
    FirstName VARCHAR(20),
    LastName VARCHAR(50)
);
INSERT INTO EmployeeNames (FirstName, LastName)
SELECT FirstName, LastName
FROM Employee
""")

In [ ]:
q("""
SELECT * FROM EmployeeNames
""")

### 3.2 UPDATE — modifying existing data

Update one specific row:

In [ ]:
q("""
UPDATE Employee
SET LastName = 'Akbari'
WHERE EmployeeID = 1
""")

In [ ]:
q("""
SELECT * FROM Employee WHERE EmployeeID = 1
""")

Apply a change to all rows in a column:

In [ ]:
q("""
UPDATE Employee
SET Age = Age + 10
""")

In [ ]:
q("""
SELECT * FROM Employee
""")

> ⚠️ **Always use WHERE with UPDATE.** Without it, UPDATE changes every row (like the Age+10 above — which we did on purpose!). There is no Undo.

### 3.3 DELETE — removing rows

In [ ]:
q("""
DELETE FROM Employee
WHERE EmployeeID = 4
""")

In [ ]:
q("""
SELECT * FROM Employee
""")

> ⚠️ **Always use WHERE with DELETE.** Omitting it deletes every row in the table.

## Part 4 — Real-World Use Cases

### 4.1 Archiving old data

As a database grows, old records are moved to archive tables. Step 1: copy old rows to the archive (parent first). Step 2: delete them from the live table (child first).

Let's add some sales data to work with:

In [ ]:
q("""
INSERT INTO Customer VALUES (1,'Sara','Lopez'), (2,'Amin','Amirkhalili');
INSERT INTO Sale VALUES
 (1, '2013-05-10', 2.10, 5.00, 1),
 (2, '2013-11-02', 3.30, 5.00, 2),
 (3, '2015-06-20', 4.10, 0.00, 1),
 (4, '2016-01-15', 1.90, 5.00, 2);
INSERT INTO SaleItem VALUES
 (1, 101, 9.0, 1, 80.00),
 (2, 102, 8.0, 2, 45.00),
 (3, 103, 10.0, 1, 120.00),
 (4, 101, 9.5, 1, 85.00)
""")

**Step 1a — archive the parent table first:**

In [ ]:
q("""
DROP TABLE IF EXISTS SaleArchive;
CREATE TABLE SaleArchive AS SELECT * FROM Sale WHERE 0;
INSERT INTO SaleArchive
SELECT *
FROM Sale
WHERE SaleDate < '2014-01-01'
""")

**Step 1b — archive the child table, linking back to the parent via a join** (SaleItem has no date column, so we look up the date from Sale):

In [ ]:
q("""
DROP TABLE IF EXISTS SaleItemArchive;
CREATE TABLE SaleItemArchive AS SELECT * FROM SaleItem WHERE 0;
INSERT INTO SaleItemArchive
SELECT si.*
FROM SaleItem si
JOIN Sale s ON si.SaleID = s.SaleID
WHERE s.SaleDate < '2014-01-01'
""")

In [ ]:
q("""
SELECT * FROM SaleArchive
""")

In [ ]:
q("""
SELECT * FROM SaleItemArchive
""")

### 4.2 Deleting old data after archiving

Always delete in reverse order — child tables first, then parent:

In [ ]:
q("""
DELETE FROM SaleItem
WHERE SaleID IN (
    SELECT SaleID
    FROM Sale
    WHERE SaleDate < '2014-01-01'
)
""")

In [ ]:
q("""
DELETE FROM Sale
WHERE SaleDate < '2014-01-01'
""")

In [ ]:
q("""
SELECT * FROM Sale
""")

### 4.3 Applying a business rule update

When a business rule changes — such as a price increase for a specific group of products — UPDATE combined with a subquery applies the change precisely:

```sql
UPDATE Product
SET ListPrice = ListPrice * 1.05
WHERE ManufacturerID IN (
    SELECT ManufacturerID
    FROM Manufacturer
    WHERE State = 'NH'
);
```

*(Our practice database has no Product/Manufacturer tables yet — creating them and running this update is a great exercise!)*

## Summary — Key Commands at a Glance

| Category | Command | Description |
|---|---|---|
| DDL | CREATE TABLE | Create a new table with columns and constraints. |
| DDL | DROP TABLE | Permanently delete a table (child before parent). |
| DDL | SELECT ... INTO / CREATE TABLE AS | Create a new table by copying data. |
| DDL | ALTER TABLE ADD | Add a new column or key. |
| DDL | ALTER TABLE DROP COLUMN | Remove a column. |
| DDL | ALTER TABLE ALTER COLUMN | Change a column's data type (SQL Server). |
| DDL | EXEC sp_rename / RENAME TO | Rename a table. |
| DML | INSERT INTO | Add new rows. |
| DML | INSERT INTO ... SELECT | Copy rows between tables. |
| DML | UPDATE | Modify existing rows (always use WHERE). |
| DML | DELETE | Remove specific rows (always use WHERE). |

---
## 🎯 Your Turn — Build Your Own Tables

Now practice everything from this notebook yourself. Write your SQL inside the `q( ... )` helper in the empty cells below. Use `tables()` any time to see what exists.

**Exercise 1 — Create a parent table.** Create a `Manufacturer` table with `ManufacturerID` (INT, primary key), `ManufacturerName` (VARCHAR(50), NOT NULL), and `State` (VARCHAR(2)).

In [ ]:
q("""
-- your CREATE TABLE Manufacturer here

""")

**Exercise 2 — Create a child table with a foreign key.** Create a `Product` table with `ProductID` (primary key), `ProductName`, `ListPrice` (NUMERIC(8,2)), and `ManufacturerID` with a FOREIGN KEY referencing `Manufacturer(ManufacturerID)`.

In [ ]:
q("""
-- your CREATE TABLE Product here

""")

**Exercise 3 — Insert data.** Insert at least 2 manufacturers (make one of them from state 'NH') and 3 products. Then SELECT both tables to check.

In [ ]:
q("""
-- your INSERT statements here

""")

In [ ]:
try:
    display(q("SELECT * FROM Manufacturer"))
except Exception:
    print("👉 Nothing here yet — create the Manufacturer table first (Exercise 1).")

In [ ]:
try:
    display(q("SELECT * FROM Product"))
except Exception:
    print("👉 Nothing here yet — create the Product table first (Exercise 2).")

**Exercise 4 — Referential integrity in action.** Try inserting a product with a `ManufacturerID` that does not exist. The database should reject it — that is the foreign key doing its job.

In [ ]:
try:
    q("INSERT INTO Product VALUES (99, 'GhostShoe', 10.0, 12345)")
except Exception as e:
    print("❌ Rejected, as expected:", e)

**Exercise 5 — Apply the business rule update from Part 4.3.** Increase the price of all products made by manufacturers in New Hampshire by 5%, using UPDATE with a subquery. Then SELECT Product to confirm only NH products changed.

In [ ]:
q("""
-- your UPDATE ... WHERE ManufacturerID IN (SELECT ...) here

""")

In [ ]:
try:
    display(q("SELECT * FROM Product"))
except Exception:
    print("👉 Nothing here yet — create the Product table first (Exercise 2).")

**Challenge — build a mini schema from scratch.** Create `Customer`, `OrderTbl` (with a FK to Customer), and `OrderItem` (composite primary key, FKs to OrderTbl and Product). Insert a few rows, then try to DROP the tables in the wrong order (parent first) and see what happens.

<details><summary>💡 Click for a hint</summary>

Drop order must be child → parent: <code>OrderItem</code>, then <code>OrderTbl</code>, then <code>Customer</code>. Dropping <code>Customer</code> first fails while <code>OrderTbl</code> still references it.
</details>

In [ ]:
q("""
-- your challenge schema here

""")

---
⏭️ **Coming up in Notebook 3.1:** querying data with SELECT, FROM, WHERE, and ORDER BY.